# 世界模型前瞻规划 — 训练与评估 Notebook

**负责人**: 赵中赐  
**用途**: 在 Notebook 容器 / DCU 服务器上完成世界模型全流程训练

---

## 0. 环境检查

In [ ]:
import sys, os, numpy as np
sys.path.insert(0, os.getcwd())

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA device count: {torch.cuda.device_count()}")

# 检查数据文件
data_path = "data/raw/push_data.npz"
if os.path.exists(data_path):
    d = np.load(data_path)
    print(f"\nData: {data_path}")
    print(f"  states:      {d['states'].shape}")
    print(f"  actions:     {d['actions'].shape}")
    print(f"  next_states: {d['next_states'].shape}")
else:
    print(f"\nWARNING: {data_path} not found! 需要先运行 collect_data.py")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

## 1. 数据加载与可视化

In [ ]:
import matplotlib.pyplot as plt
from src.world_model.dataset import PushGraspDataset

dataset = PushGraspDataset(data_path="data/raw/push_data.npz")
print(f"Total samples: {len(dataset)}")

# 查看样本
s, a, ns = dataset[0]
print(f"State:      {s.numpy()}")
print(f"Action:     {a.numpy()}")
print(f"Next state: {ns.numpy()}")

# 位移分布
all_states = dataset.states.numpy()
all_next   = dataset.next_states.numpy()
disp = all_next[:, 0:2] - all_states[:, 0:2]
dist = np.linalg.norm(disp, axis=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(dist, bins=50, edgecolor="black")
axes[0].set_xlabel("Displacement (mm)")
axes[0].set_ylabel("Count")
axes[0].set_title("Target Displacement Distribution")
axes[1].scatter(disp[:, 0], disp[:, 1], s=2, alpha=0.5)
axes[1].set_xlabel("dx (mm)")
axes[1].set_ylabel("dy (mm)")
axes[1].set_title("Displacement Scatter")
axes[1].axhline(0, color="gray", ls="--")
axes[1].axvline(0, color="gray", ls="--")
axes[2].scatter(all_states[:, 0], all_states[:, 1], s=2, c=dist, cmap="viridis", alpha=0.5)
axes[2].set_xlabel("target_x (mm)")
axes[2].set_ylabel("target_y (mm)")
axes[2].set_title("State Coverage (colored by displacement)")
plt.tight_layout()
os.makedirs("experiments", exist_ok=True)
plt.savefig("experiments/data_distribution.png", dpi=150)
plt.show()

## 2. 训练 MLP 世界模型

In [ ]:
from src.utils.config import load_config
from src.world_model.trainer import train_world_model

# 加载并修改配置为 MLP 模式
config_mlp = load_config("config/world_model.yaml")
config_mlp["model"]["type"] = "mlp"
print(f"Model type: {config_mlp['model']['type']}")
print(f"Hidden dim: {config_mlp['model']['hidden_dim']}")
print(f"Epochs:     {config_mlp['training']['epochs']}")
print(f"Batch size: {config_mlp['training']['batch_size']}")

# 训练
model_mlp = train_world_model(
    config_mlp,
    data_path="data/raw/push_data.npz",
    save_dir="models/world_model/mlp",
    device=DEVICE,
)
print("MLP training complete!")

## 3. 训练 GRU 世界模型

In [ ]:
config_gru = load_config("config/world_model.yaml")
config_gru["model"]["type"] = "mlp_gru"
print(f"Model type: {config_gru['model']['type']}")

model_gru = train_world_model(
    config_gru,
    data_path="data/raw/push_data.npz",
    save_dir="models/world_model/gru",
    device=DEVICE,
)
print("GRU training complete!")

## 4. 评估与对比

In [ ]:
from torch.utils.data import DataLoader, random_split
import torch.nn as nn

# 测试集
n = len(dataset)
train_n = int(n * 0.8)
val_n = int(n * 0.1)
test_n = n - train_n - val_n
_, _, test_ds = random_split(dataset, [train_n, val_n, test_n],
                              generator=torch.Generator().manual_seed(42))
test_loader = DataLoader(test_ds, batch_size=64)

criterion = nn.MSELoss()

def eval_model(model, loader, device):
    model = model.to(device)
    model.eval()
    total_loss = 0.0
    preds, truths = [], []
    with torch.no_grad():
        for s, a, ns in loader:
            s, a, ns = s.to(device), a.to(device), ns.to(device)
            if hasattr(model, "gru"):
                pred, _ = model(s, a)
            else:
                pred = model(s, a)
            total_loss += criterion(pred, ns).item() * len(s)
            preds.append(pred.cpu().numpy())
            truths.append(ns.cpu().numpy())
    return total_loss / len(loader.dataset), np.concatenate(preds), np.concatenate(truths)

mse_mlp, preds_mlp, truths = eval_model(model_mlp, test_loader, DEVICE)
mse_gru, preds_gru, _     = eval_model(model_gru, test_loader, DEVICE)

print(f"MLP Test MSE: {mse_mlp:.6f}")
print(f"GRU Test MSE: {mse_gru:.6f}")
print(f"GRU improvement: {(1 - mse_gru/mse_mlp)*100:.1f}%" if mse_gru < mse_mlp else "GRU NOT better")

## 5. 预测可视化

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# MLP
for dim, label, col in [(0, "X", 0), (1, "Y", 1), (2, "Z", 2)]:
    ax = axes[0, col]
    ax.scatter(truths[:, dim], preds_mlp[:, dim], s=3, alpha=0.4, color="#2196F3")
    lim = max(abs(truths[:, dim]).max(), abs(preds_mlp[:, dim]).max()) * 1.2
    ax.plot([-lim, lim], [-lim, lim], "k--", alpha=0.5)
    ax.set_xlabel(f"True {label}")
    ax.set_ylabel(f"Pred {label}")
    ax.set_title(f"MLP — {label} dim")
    ax.set_aspect("equal")

# GRU
for dim, label, col in [(0, "X", 0), (1, "Y", 1), (2, "Z", 2)]:
    ax = axes[1, col]
    ax.scatter(truths[:, dim], preds_gru[:, dim], s=3, alpha=0.4, color="#4CAF50")
    lim = max(abs(truths[:, dim]).max(), abs(preds_gru[:, dim]).max()) * 1.2
    ax.plot([-lim, lim], [-lim, lim], "k--", alpha=0.5)
    ax.set_xlabel(f"True {label}")
    ax.set_ylabel(f"Pred {label}")
    ax.set_title(f"GRU — {label} dim")
    ax.set_aspect("equal")

plt.suptitle(f"World Model Prediction Accuracy\nMLP MSE={mse_mlp:.4f} | GRU MSE={mse_gru:.4f}",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("experiments/prediction_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. 消融实验: 训练数据量

In [ ]:
from copy import deepcopy

ratios = [0.2, 0.5, 1.0]
ablation_results = {}

for ratio in ratios:
    n_sub = int(n * ratio)
    sub_ds, _ = random_split(dataset, [n_sub, n - n_sub],
                              generator=torch.Generator().manual_seed(42))
    sub_loader = DataLoader(sub_ds, batch_size=64, shuffle=True)

    # 训练小模型
    cfg = deepcopy(config_mlp)
    cfg["training"]["epochs"] = 50  # 减轮加速
    from src.world_model.model import WorldModelMLP
    model_small = WorldModelMLP(
        state_dim=cfg["model"]["state_dim"],
        action_dim=cfg["model"]["action_dim"],
        hidden_dim=cfg["model"]["hidden_dim"],
    ).to(DEVICE)

    opt = torch.optim.AdamW(model_small.parameters(), lr=cfg["training"]["learning_rate"])
    for epoch in range(cfg["training"]["epochs"]):
        model_small.train()
        for s, a, ns in sub_loader:
            s, a, ns = s.to(DEVICE), a.to(DEVICE), ns.to(DEVICE)
            loss = criterion(model_small(s, a), ns)
            opt.zero_grad()
            loss.backward()
            opt.step()

    mse_sub, _, _ = eval_model(model_small, test_loader, DEVICE)
    ablation_results[ratio] = mse_sub
    print(f"Data ratio {ratio:.0%}: Test MSE = {mse_sub:.6f}")

# 画图
fig, ax = plt.subplots(figsize=(8, 5))
x = [int(r * 100) for r in ratios]
y = [ablation_results[r] for r in ratios]
ax.plot(x, y, "o-", linewidth=2, markersize=8, color="#2196F3")
ax.set_xlabel("Training Data (%)")
ax.set_ylabel("Test MSE")
ax.set_title("Data Size Ablation")
ax.grid(True, alpha=0.3)
for xi, yi in zip(x, y):
    ax.annotate(f"{yi:.4f}", (xi, yi), textcoords="offset points", xytext=(0, 10))
plt.tight_layout()
plt.savefig("experiments/data_size_ablation.png", dpi=150)
plt.show()

## 7. 导出模型权重

In [ ]:
import torch

# 保存最终模型 (选择更优者)
best_model = model_gru if mse_gru < mse_mlp else model_mlp
torch.save(best_model.state_dict(), "models/world_model/world_model.pt")
print(f"Best model saved: models/world_model/world_model.pt")
print(f"  Type: {'GRU' if mse_gru < mse_mlp else 'MLP'}")
print(f"  Test MSE: {min(mse_mlp, mse_gru):.6f}")

# 同时单独保存两个模型供对比
os.makedirs("models/world_model/mlp", exist_ok=True)
os.makedirs("models/world_model/gru", exist_ok=True)
torch.save(model_mlp.state_dict(), "models/world_model/mlp/world_model.pt")
torch.save(model_gru.state_dict(), "models/world_model/gru/world_model.pt")
print("Both models saved separately.")

## 8. 生成 Loss 曲线 (从 TensorBoard)

在终端运行: `tensorboard --logdir models/world_model/`  
或手动从 TensorBoard event 文件提取数据画图。

In [ ]:
# 从 TensorBoard event 文件提取 loss 曲线
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import glob

def extract_loss_curves(logdir, tag="Loss/train"):
    for event_file in glob.glob(f"{logdir}/**/events.out.tfevents.*", recursive=True):
        ea = EventAccumulator(event_file)
        ea.Reload()
        if tag in ea.Tags()["scalars"]:
            events = ea.Scalars(tag)
            return [e.step for e in events], [e.value for e in events]
    return [], []

fig, ax = plt.subplots(figsize=(10, 6))
for name, logdir, color in [("MLP", "models/world_model/mlp/tensorboard", "#2196F3"),
                              ("GRU", "models/world_model/gru/tensorboard", "#4CAF50")]:
    try:
        steps, vals = extract_loss_curves(logdir, "Loss/train")
        if steps:
            ax.plot(steps, vals, color=color, label=f"{name} train", alpha=0.7)
        steps, vals = extract_loss_curves(logdir, "Loss/val")
        if steps:
            ax.plot(steps, vals, color=color, label=f"{name} val", ls="--", alpha=0.7)
    except Exception as e:
        print(f"{name} TensorBoard not found: {e}")

ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Training Curves: MLP vs GRU")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("experiments/loss_curves.png", dpi=150)
plt.show()
print("\n训练完成! 所有图表已保存到 experiments/")